# 02 — Silver

Typed, flagged, **unjoined** tables. **No SGI number here.** Gold computes SGI-30 from tickets.

| Table | Grain | Job |
|---|---|---|
| `silver_rat_sightings` | one ticket (`unique_key`) | types, ZIP, flags, GPS cluster, same-timestamp close |
| `silver_restaurant_inspections` | one **violation** | types, flags; do not `SUM(score)` |
| `silver_inspection_visits` | one **visit** (`camis`,`inspection_dt`) | score window; 04K/04L rodent; 08A harborage separate |
| `silver_zip_spine` | one ZIP | borough, `zip_type`, cluster share (N≥30 gate) |
| `silver_dropped_non_nyc` | one dropped row | audit log of NJ / Long Island / Westchester rows removed from silver |

**Geography:** the deliverable is a Service Gap Index for **NYC** ZIP codes. Rows whose ZIP is outside the five boroughs (New Jersey `07xxx`/`08xxx`, Nassau/Suffolk `115xx`/`117xx`–`119xx`, Westchester/Rockland `105xx`–`109xx`) are **removed** from every silver table by `workspace.default.zip_region()` and logged in `silver_dropped_non_nyc`. In this packet that is 80 kitchen rows / 15 establishments / 24 visits (all `boro = 0`, e.g. commissary or HQ addresses) and **zero** 311 tickets. `110xx` straddles Queens and Nassau, so it is NYC only when the source row carries an NYC borough label (all `110xx` rows in the packet are labelled QUEENS). Blank ZIP rows are **kept** — no ZIP is not the same as out of state.

**Missing data:** keep the row. Fill borough from ZIP **only** when that ZIP has exactly one NYC borough in the packet. Never invent a ZIP. Never treat `Closed` as inspected.

`closed_ts` is a clock, not show-up. `is_same_timestamp_close` flags ~22,547 instant closes. Resolution text is not in the packet.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_zip_boro_lookup AS
WITH src AS (
  SELECT
    CASE
      WHEN TRIM(incident_zip) RLIKE '^[0-9]{5}$' AND TRIM(incident_zip) != '12345'
        THEN TRIM(incident_zip)
    END AS zip,
    UPPER(TRIM(borough)) AS borough,
    COUNT(*) AS n
  FROM workspace.default.bronze_rat_sightings
  WHERE TRIM(incident_zip) RLIKE '^[0-9]{5}$'
    AND TRIM(incident_zip) != '12345'
    AND UPPER(TRIM(borough)) IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND')
  GROUP BY 1, 2
  UNION ALL
  SELECT
    CASE
      WHEN TRIM(zipcode) RLIKE '^[0-9]{5}$' AND TRIM(zipcode) != '12345'
        THEN TRIM(zipcode)
    END AS zip,
    UPPER(TRIM(boro)) AS borough,
    COUNT(*) AS n
  FROM workspace.default.bronze_restaurant_inspections
  WHERE TRIM(zipcode) RLIKE '^[0-9]{5}$'
    AND TRIM(zipcode) != '12345'
    AND UPPER(TRIM(boro)) IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND')
  GROUP BY 1, 2
),
agg AS (
  SELECT zip, borough, SUM(n) AS n
  FROM src
  WHERE zip IS NOT NULL
  GROUP BY zip, borough
)
SELECT
  zip,
  borough,
  n,
  SUM(n) OVER (PARTITION BY zip) AS n_in_zip,
  COUNT(*) OVER (PARTITION BY zip) AS n_distinct_borough,
  ROW_NUMBER() OVER (PARTITION BY zip ORDER BY n DESC, borough) AS rn
FROM agg;

COMMENT ON TABLE workspace.default.silver_zip_boro_lookup IS
  'Helper: ZIP to NYC borough counts from the two packet files only. 10463 and 11370 are split. Not a Genie table.';
COMMENT ON COLUMN workspace.default.silver_zip_boro_lookup.zip IS
  'Five-digit ZIP observed in packet 311 or kitchens. Not an authoritative USPS or ZCTA dimension.';
COMMENT ON COLUMN workspace.default.silver_zip_boro_lookup.borough IS
  'NYC borough on source rows for this ZIP. Split ZIPs have two rows (10463 Marble Hill, 11370 Rikers / East Elmhurst).';
COMMENT ON COLUMN workspace.default.silver_zip_boro_lookup.n IS
  'Packet rows contributing this ZIP-borough pair.';
COMMENT ON COLUMN workspace.default.silver_zip_boro_lookup.n_in_zip IS
  'Total packet rows for this ZIP across boroughs.';
COMMENT ON COLUMN workspace.default.silver_zip_boro_lookup.n_distinct_borough IS
  '1 = unique borough, safe to fill. 2 = ambiguous; do not silently pick a borough for metrics that need a single borough.';
COMMENT ON COLUMN workspace.default.silver_zip_boro_lookup.rn IS
  '1 = modal borough by count. Join lookup on zip AND rn = 1, then still check n_distinct_borough.';

In [0]:
%sql
-- Single source of truth for "is this ZIP in New York City". Prefix-based, not lookup-based,
-- so an out-of-state ZIP cannot sneak in just because a source row was mislabelled with a borough.
--   100-102 Manhattan | 103 Staten Island | 104 Bronx | 111/113/114/116 Queens | 112 Brooklyn
--   110 straddles Queens (11004, 11005, parts of 11001/11040) and Nassau -> NYC only with an NYC borough label
--   07/08 New Jersey | 115/117/118/119 Nassau-Suffolk (Long Island) | 105-109 Westchester / Rockland
CREATE OR REPLACE FUNCTION workspace.default.zip_region(zip STRING, borough STRING)
RETURNS STRING
COMMENT 'Classifies a normalized 5-digit ZIP: nyc / new_jersey / long_island / westchester_rockland / other_non_nyc. NULL zip returns NULL. 110xx is nyc only when the source borough is an NYC borough (Queens/Nassau straddle). Silver keeps only nyc and NULL-zip rows; everything else is logged in silver_dropped_non_nyc.'
RETURN CASE
  WHEN zip IS NULL THEN NULL
  WHEN SUBSTR(zip, 1, 3) IN ('100','101','102','103','104','111','112','113','114','116') THEN 'nyc'
  WHEN SUBSTR(zip, 1, 3) = '110'
       AND UPPER(TRIM(borough)) IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND') THEN 'nyc'
  WHEN SUBSTR(zip, 1, 3) = '110' THEN 'long_island'
  WHEN SUBSTR(zip, 1, 2) IN ('07','08') THEN 'new_jersey'
  WHEN SUBSTR(zip, 1, 3) IN ('115','117','118','119') THEN 'long_island'
  WHEN SUBSTR(zip, 1, 3) IN ('105','106','107','108','109') THEN 'westchester_rockland'
  ELSE 'other_non_nyc'
END;

-- Quarantine: every bronze row that silver drops for being outside NYC. Log it, do not impute it.
CREATE OR REPLACE TABLE workspace.default.silver_dropped_non_nyc AS
WITH raw AS (
  SELECT
    'restaurant_inspections.csv' AS _source_file,
    NULLIF(TRIM(camis), '') AS record_id,
    NULLIF(TRIM(dba), '') AS record_name,
    NULLIF(TRIM(boro), '') AS borough_raw,
    NULLIF(TRIM(zipcode), '') AS zip_raw
  FROM workspace.default.bronze_restaurant_inspections
  UNION ALL
  SELECT
    'rat_sightings.csv',
    unique_key,
    NULLIF(TRIM(descriptor), ''),
    NULLIF(TRIM(borough), ''),
    NULLIF(TRIM(incident_zip), '')
  FROM workspace.default.bronze_rat_sightings
),
normalized AS (
  SELECT
    *,
    CASE
      WHEN zip_raw RLIKE '^[0-9]{5}$' AND zip_raw != '12345' THEN zip_raw
      WHEN zip_raw RLIKE '^[0-9]{5}-[0-9]{4}$' AND SUBSTR(zip_raw, 1, 5) != '12345' THEN SUBSTR(zip_raw, 1, 5)
      WHEN zip_raw RLIKE '^[0-9]{5}\\.0$' AND SUBSTR(zip_raw, 1, 5) != '12345' THEN SUBSTR(zip_raw, 1, 5)
      ELSE NULL
    END AS zip
  FROM raw
),
regioned AS (
  SELECT *, workspace.default.zip_region(zip, borough_raw) AS zip_region
  FROM normalized
)
SELECT
  _source_file, record_id, record_name, borough_raw, zip_raw, zip, zip_region,
  current_timestamp() AS _ingested_at
FROM regioned
WHERE zip IS NOT NULL AND zip_region <> 'nyc';

COMMENT ON TABLE workspace.default.silver_dropped_non_nyc IS
  'Audit log of bronze rows REMOVED from silver because the ZIP is outside New York City (New Jersey, Long Island Nassau/Suffolk, Westchester/Rockland). Grain: one bronze row (violation row for kitchens, ticket for 311). In this packet: 80 kitchen rows / 15 establishments, all boro = 0 — NYC-permitted vendors with an out-of-state commissary or HQ address — and 0 311 tickets. Not a Genie table for ranking; use it to answer what we excluded and why. Blank-ZIP rows are NOT here; they stay in silver with zip = NULL.';
COMMENT ON COLUMN workspace.default.silver_dropped_non_nyc._source_file IS
  'restaurant_inspections.csv or rat_sightings.csv.';
COMMENT ON COLUMN workspace.default.silver_dropped_non_nyc.record_id IS
  'camis for kitchens, unique_key for 311. Count establishments with COUNT(DISTINCT record_id) WHERE _source_file = restaurant_inspections.csv.';
COMMENT ON COLUMN workspace.default.silver_dropped_non_nyc.record_name IS
  'dba (trade name) for kitchens, 311 descriptor for tickets.';
COMMENT ON COLUMN workspace.default.silver_dropped_non_nyc.borough_raw IS
  'Borough as delivered. 0 on every dropped kitchen row: DOHMH has no borough for an out-of-city address.';
COMMENT ON COLUMN workspace.default.silver_dropped_non_nyc.zip_raw IS
  'ZIP string as delivered in the packet.';
COMMENT ON COLUMN workspace.default.silver_dropped_non_nyc.zip IS
  'Normalized 5-digit ZIP that triggered the drop.';
COMMENT ON COLUMN workspace.default.silver_dropped_non_nyc.zip_region IS
  'new_jersey / long_island / westchester_rockland / other_non_nyc from workspace.default.zip_region(). Never nyc in this table.';
COMMENT ON COLUMN workspace.default.silver_dropped_non_nyc._ingested_at IS
  'Table rebuild timestamp.';

In [0]:
%sql
SELECT
  _source_file,
  zip_region,
  zip,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT record_id) AS n_ids,
  ARRAY_JOIN(ARRAY_SORT(COLLECT_SET(record_name)), ' | ') AS names
FROM workspace.default.silver_dropped_non_nyc
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;

_source_file,zip_region,zip,n_rows,n_ids,names
restaurant_inspections.csv,long_island,11542,4,1,SONY CAFE
restaurant_inspections.csv,long_island,11701,15,1,BAR TABAC
restaurant_inspections.csv,long_island,11762,2,1,FRENCH LOUIE
restaurant_inspections.csv,new_jersey,07002,3,2,LAZYDOG | WAFELS AND DINGES
restaurant_inspections.csv,new_jersey,07071,2,1,SOUTH SLOPE RESTAURANT & BAR
restaurant_inspections.csv,new_jersey,07304,5,1,BROOKLYN CRAB
restaurant_inspections.csv,new_jersey,07307,18,3,CAFFEINE UNDERGROUND | MADELINE'S | SERENECO
restaurant_inspections.csv,new_jersey,07627,2,1,POWER BOWLS
restaurant_inspections.csv,new_jersey,07631,12,1,AROMA HAVEN CAFE
restaurant_inspections.csv,new_jersey,08550,12,1,LAUNDRY & LATTE


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_rat_sightings AS
WITH parsed AS (
  SELECT
    unique_key,
    TRY_CAST(NULLIF(TRIM(created_date), '') AS TIMESTAMP) AS created_ts,
    NULLIF(TRIM(closed_date), '') AS closed_date_raw,
    TRY_CAST(NULLIF(TRIM(closed_date), '') AS TIMESTAMP) AS closed_ts,
    NULLIF(TRIM(status), '') AS status,
    NULLIF(TRIM(complaint_type), '') AS complaint_type,
    NULLIF(TRIM(descriptor), '') AS descriptor,
    NULLIF(TRIM(location_type), '') AS location_type_raw,
    NULLIF(TRIM(incident_zip), '') AS zip_raw,
    NULLIF(TRIM(borough), '') AS borough_raw,
    TRY_CAST(NULLIF(TRIM(latitude), '') AS DOUBLE) AS lat,
    TRY_CAST(NULLIF(TRIM(longitude), '') AS DOUBLE) AS lon
  FROM workspace.default.bronze_rat_sightings
),
normalized AS (
  SELECT
    unique_key, created_ts, closed_ts, status, complaint_type, descriptor,
    borough_raw, lat, lon, zip_raw,
    CASE
      WHEN location_type_raw RLIKE 'Catch Basin' THEN 'Catch Basin/Sewer'
      WHEN location_type_raw RLIKE 'Parking Lot' THEN 'Parking Lot'
      WHEN location_type_raw RLIKE 'Day Care' THEN 'Day Care'
      WHEN location_type_raw RLIKE 'School' THEN 'School'
      ELSE location_type_raw
    END AS location_type,
    CASE
      WHEN zip_raw RLIKE '^[0-9]{5}$' AND zip_raw != '12345' THEN zip_raw
      WHEN zip_raw RLIKE '^[0-9]{5}-[0-9]{4}$' AND SUBSTR(zip_raw, 1, 5) != '12345' THEN SUBSTR(zip_raw, 1, 5)
      WHEN zip_raw RLIKE '^[0-9]{5}\\.0$' AND SUBSTR(zip_raw, 1, 5) != '12345' THEN SUBSTR(zip_raw, 1, 5)
      ELSE NULL
    END AS zip,
    CASE
      WHEN zip_raw IS NULL THEN 'blank'
      WHEN zip_raw = '12345' THEN 'suspicious_12345'
      WHEN zip_raw RLIKE '^[0-9]{5}(-[0-9]{4})?$' OR zip_raw RLIKE '^[0-9]{5}\\.0$' THEN 'valid'
      ELSE 'invalid_format'
    END AS dq_zip_flag,
    CASE
      WHEN lat IS NULL OR lon IS NULL THEN 'missing'
      WHEN lat < 40.4 OR lat > 41.0 OR lon < -74.5 OR lon > -73.5 THEN 'out_of_domain'
      ELSE 'valid'
    END AS dq_coords_flag,
    CASE
      WHEN closed_date_raw IS NOT NULL AND closed_ts IS NULL THEN 'parse_fail'
      WHEN closed_ts IS NOT NULL AND created_ts IS NOT NULL AND closed_ts < created_ts THEN 'negative_duration'
      ELSE 'valid'
    END AS dq_closed_flag,
    CASE
      WHEN lat IS NULL OR lon IS NULL THEN NULL
      ELSE CONCAT(CAST(ROUND(lat, 4) AS STRING), '|', CAST(ROUND(lon, 4) AS STRING))
    END AS cluster_key,
    CASE
      WHEN descriptor = 'Rat Sighting' THEN 'rat_sighting'
      WHEN descriptor = 'Mouse Sighting' THEN 'mouse_sighting'
      WHEN descriptor = 'Signs of Rodents' THEN 'signs_of_rodents'
      WHEN descriptor = 'Condition Attracting Rodents' THEN 'condition_attracting_rodents'
      ELSE 'unknown'
    END AS descriptor_category,
    CASE
      WHEN location_type IN ('3+ Family Apt. Building','1-2 Family Dwelling','3+ Family Mixed Use Building','1-2 Family Mixed Use Building','Single Room Occupancy (SRO)') THEN 'dwelling'
      WHEN location_type IN ('Sidewalk','Street','Public Stairs','Public Garden') THEN 'public_space'
      WHEN location_type IN ('Commercial Building','Office Building','Parking Lot') THEN 'commercial'
      WHEN location_type IN ('Catch Basin/Sewer','Construction Site') THEN 'infrastructure'
      WHEN location_type IN ('School','Day Care','Hospital','Government Building','Summer Camp') THEN 'institution'
      WHEN location_type IN ('Vacant Lot','Vacant Building') THEN 'vacant'
      ELSE 'other'
    END AS location_category
  FROM parsed
),
-- Geography gate: drop NJ / Long Island / Westchester ZIPs (logged in silver_dropped_non_nyc).
-- Blank / quarantined ZIP (zip IS NULL) is kept: unknown location is not out of state.
nyc_only AS (
  SELECT *
  FROM normalized
  WHERE zip IS NULL
     OR workspace.default.zip_region(zip, borough_raw) = 'nyc'
),
with_lookup AS (
  SELECT
    n.*,
    l.borough AS borough_from_zip,
    l.n_distinct_borough
  FROM nyc_only n
  LEFT JOIN workspace.default.silver_zip_boro_lookup l
    ON n.zip = l.zip AND l.rn = 1
),
filled AS (
  SELECT
    unique_key, created_ts, closed_ts, status, complaint_type, descriptor,
    descriptor_category, location_type, location_category, zip, lat, lon, cluster_key,
    dq_zip_flag, dq_coords_flag, dq_closed_flag,
    CASE
      WHEN UPPER(borough_raw) IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND')
        THEN UPPER(borough_raw)
      WHEN n_distinct_borough = 1 THEN borough_from_zip
      ELSE NULL
    END AS borough,
    CASE
      WHEN UPPER(borough_raw) IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND')
        THEN 'source'
      WHEN n_distinct_borough = 1 THEN 'zip_lookup'
      WHEN n_distinct_borough > 1 THEN 'ambiguous_zip'
      ELSE 'unknown'
    END AS borough_source,
    CASE
      WHEN zip IS NULL THEN 'unknown'
      WHEN zip IN ('11430', '11371') THEN 'airport'
      WHEN zip = '10128' THEN 'neighborhood'
      WHEN zip LIKE '101%' THEN 'building'
      WHEN n_distinct_borough IS NULL THEN 'non_nyc'
      ELSE 'neighborhood'
    END AS zip_type,
    CASE
      WHEN closed_ts IS NOT NULL AND created_ts IS NOT NULL AND closed_ts = created_ts THEN TRUE
      ELSE FALSE
    END AS is_same_timestamp_close,
    CASE
      WHEN closed_ts IS NOT NULL AND created_ts IS NOT NULL AND DATEDIFF(closed_ts, created_ts) = 0 THEN TRUE
      ELSE FALSE
    END AS is_same_calendar_close
  FROM with_lookup
),
clustered AS (
  SELECT
    f.*,
    CASE
      WHEN cluster_key IS NULL THEN NULL
      ELSE COUNT(*) OVER (PARTITION BY zip, cluster_key)
    END AS cluster_n
  FROM filled f
),
deduped AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY unique_key ORDER BY created_ts) AS rn
  FROM clustered
)
SELECT
  unique_key, created_ts, closed_ts, status, complaint_type, descriptor,
  descriptor_category, location_type, location_category, zip, zip_type, borough, borough_source,
  lat, lon, cluster_key, cluster_n,
  dq_zip_flag, dq_coords_flag, dq_closed_flag,
  is_same_timestamp_close, is_same_calendar_close,
  'rat_sightings.csv' AS _source_file,
  current_timestamp() AS _ingested_at
FROM deduped
WHERE rn = 1;

COMMENT ON TABLE workspace.default.silver_rat_sightings IS
  'Cleaned 311 rodent tickets. One row per unique_key. Closed is not inspected — no resolution text. is_same_timestamp_close flags instant closes (~22547). descriptor_category splits four descriptors (indoor mice ≠ street rats). location_category groups 22 types (dwelling ~77.5pct). ZIP 12345 quarantined. Borough filled from ZIP only when unique in-packet. cluster_key is ROUND(lat,4)|ROUND(lon,4); 10035 is mostly one GPS. Not joined to kitchens. Gold computes SGI-30 from this table, not from Closed percent.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.unique_key IS
  '311 request ID. Grain of this table: one ticket, not one rat, caller, property, or unique incident.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.created_ts IS
  'Ticket created timestamp. Source has no timezone offset; treat as NYC wall-clock. Use for cohorts. Do not use closed_ts to choose the creation denominator.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.closed_ts IS
  'Administrative closure timestamp, not proof of a visit, inspection, or extermination. Resolution text is not in the packet. NULL if still open.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.status IS
  'Administrative status: Closed, In Progress, or Unspecified. Closed is not the city showed up.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.complaint_type IS
  'All rows are Rodent (DOHMH). Not an HPD housing complaint. Default demand scope is all four descriptors under Rodent.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.descriptor IS
  'Raw 311 descriptor. Prefer descriptor_category for grouping.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.descriptor_category IS
  'rat_sighting / mouse_sighting / signs_of_rodents / condition_attracting_rodents. Indoor mice are not street rats. Keep all four in demand; do not match 04K kitchens to mouse-in-apt 311.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.location_type IS
  'Reported setting after merging 4 naming variants (22 types). 3+ Family Apt is about half of tickets: density / landlord-fear hint, not NYCHA proof.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.location_category IS
  'dwelling / public_space / commercial / infrastructure / institution / vacant / other. About 77.5pct dwelling. Show the split on a ZIP page; do not average a sewer and a bedroom.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.zip IS
  'Normalized 5-digit ZIP. 12345 quarantined to NULL. This is a ZIP, not a block or neighborhood boundary.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.zip_type IS
  'Ticket-level: neighborhood / airport (11430 JFK, 11371 LGA) / building (101xx except 10128) / unknown. 10128 is Upper East Side housing, not a building ZIP. non_nyc is a guard only — NJ / Long Island / Westchester tickets are already removed by zip_region() and logged in silver_dropped_non_nyc (0 in this packet). Use silver_zip_spine.zip_type for ranking (adds thin when n_311 < 10).';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.borough IS
  'BRONX, BROOKLYN, MANHATTAN, QUEENS, or STATEN ISLAND. Filled from ZIP only when that ZIP has exactly one NYC borough in the packet.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.borough_source IS
  'source / zip_lookup / ambiguous_zip / unknown. zip_lookup is the 1 Unspecified+11237 fill and similar; not a geocoder.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.lat IS
  'Request latitude, not a restaurant coordinate. Missing on 94 rows. Do not rewrite ZIP from coords. No BBL in the packet.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.lon IS
  'Request longitude, not a restaurant coordinate. Out-of-domain coords are flagged; they do not drop the ticket from ZIP totals.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.cluster_key IS
  'ROUND(lat,4)|ROUND(lon,4). Shared GPS can be snapping, one campus, or one interpolator. Say one GPS cluster, not one lot.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.cluster_n IS
  'Tickets in this ZIP with the same cluster_key. 10035 is mostly one cluster (~1227 of 1501).';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.dq_zip_flag IS
  'valid / suspicious_12345 / blank / invalid_format. 12345 is quarantined.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.dq_coords_flag IS
  'valid / missing / out_of_domain. Bad coords affect maps, not otherwise valid ZIP totals.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.dq_closed_flag IS
  'valid / parse_fail / negative_duration. Exclude parse_fail and negative_duration from SGI-30 N and K.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.is_same_timestamp_close IS
  'TRUE when closed_ts equals created_ts (~22547 tickets). A fast close may be a visit, duplicate, no access, or bounce. Do not treat as show-up. Do not put Closed percent in SGI.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings.is_same_calendar_close IS
  'TRUE when closed and created fall on the same calendar date. Broader than same-timestamp; still not a visit.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings._source_file IS
  'Packet filename: rat_sightings.csv.';
COMMENT ON COLUMN workspace.default.silver_rat_sightings._ingested_at IS
  'Table rebuild timestamp. Not the 311 observation cutoff. Gold cutoff is MAX(created_ts), not current_timestamp().';

In [0]:
%sql
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT unique_key) AS n_distinct_keys,
  SUM(CASE WHEN zip IS NULL THEN 1 ELSE 0 END) AS n_null_zip,
  SUM(CASE WHEN dq_zip_flag = 'suspicious_12345' THEN 1 ELSE 0 END) AS n_12345,
  SUM(CASE WHEN borough_source = 'zip_lookup' THEN 1 ELSE 0 END) AS n_boro_from_zip,
  SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) AS n_null_borough,
  SUM(CASE WHEN dq_coords_flag = 'missing' THEN 1 ELSE 0 END) AS n_missing_coords,
  SUM(CASE WHEN dq_coords_flag = 'out_of_domain' THEN 1 ELSE 0 END) AS n_bad_coords,
  SUM(CASE WHEN zip_type = 'airport' THEN 1 ELSE 0 END) AS n_airport,
  SUM(CASE WHEN zip_type = 'building' THEN 1 ELSE 0 END) AS n_building,
  COUNT(DISTINCT location_type) AS n_location_types,
  SUM(CASE WHEN descriptor_category = 'rat_sighting' THEN 1 ELSE 0 END) AS n_rat_sighting,
  SUM(CASE WHEN descriptor_category = 'mouse_sighting' THEN 1 ELSE 0 END) AS n_mouse_sighting,
  SUM(CASE WHEN descriptor_category = 'signs_of_rodents' THEN 1 ELSE 0 END) AS n_signs_rodents,
  SUM(CASE WHEN descriptor_category = 'condition_attracting_rodents' THEN 1 ELSE 0 END) AS n_condition_attracting,
  SUM(CASE WHEN location_category = 'dwelling' THEN 1 ELSE 0 END) AS n_dwelling,
  SUM(CASE WHEN location_category = 'public_space' THEN 1 ELSE 0 END) AS n_public_space,
  SUM(CASE WHEN location_category = 'commercial' THEN 1 ELSE 0 END) AS n_commercial,
  SUM(CASE WHEN location_category = 'infrastructure' THEN 1 ELSE 0 END) AS n_infrastructure,
  SUM(CASE WHEN location_category = 'institution' THEN 1 ELSE 0 END) AS n_institution,
  SUM(CASE WHEN location_category = 'vacant' THEN 1 ELSE 0 END) AS n_vacant,
  SUM(CASE WHEN location_category = 'other' THEN 1 ELSE 0 END) AS n_other_loc,
  SUM(CASE WHEN is_same_timestamp_close THEN 1 ELSE 0 END) AS n_same_timestamp_close,
  SUM(CASE WHEN is_same_calendar_close THEN 1 ELSE 0 END) AS n_same_calendar_close,
  MIN(created_ts) AS created_min,
  MAX(created_ts) AS created_max
FROM workspace.default.silver_rat_sightings;

n_rows,n_distinct_keys,n_null_zip,n_12345,n_boro_from_zip,n_null_borough,n_missing_coords,n_bad_coords,n_airport,n_building,n_location_types,n_rat_sighting,n_mouse_sighting,n_signs_rodents,n_condition_attracting,n_dwelling,n_public_space,n_commercial,n_infrastructure,n_institution,n_vacant,n_other_loc,n_same_timestamp_close,n_same_calendar_close,created_min,created_max
50954,50954,1,1,1,0,94,0,2,8,22,32942,2046,5178,10788,39507,3360,2695,592,371,1202,3227,22547,24691,2025-01-01T00:43:43.000Z,2026-09-17T01:35:48.000Z


## Silver — restaurant_inspections
Exact duplicate rows (identical on all fields) are collapsed — 166 extras, not missing-data rows. `boro = '0'` → NULL. ZIP blanks stay NULL. Score conflicts are **flagged**, not averaged. Grade A violation rows coincidentally equal 50,954 (the 311 ticket count) — never pitch that match.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_restaurant_inspections AS
WITH parsed AS (
  SELECT
    NULLIF(TRIM(camis), '') AS camis,
    NULLIF(TRIM(dba), '') AS dba,
    NULLIF(TRIM(boro), '') AS boro_raw,
    NULLIF(TRIM(zipcode), '') AS zip_raw,
    NULLIF(TRIM(cuisine_description), '') AS cuisine_description,
    TRY_CAST(NULLIF(TRIM(inspection_date), '') AS DATE) AS inspection_dt,
    NULLIF(TRIM(violation_code), '') AS violation_code,
    NULLIF(TRIM(violation_description), '') AS violation_description,
    NULLIF(TRIM(critical_flag), '') AS critical_flag,
    NULLIF(TRIM(score), '') AS score_raw,
    TRY_CAST(NULLIF(TRIM(score), '') AS INT) AS score_int,
    NULLIF(TRIM(grade), '') AS grade
  FROM workspace.default.bronze_restaurant_inspections
),
normalized AS (
  SELECT
    camis, dba, cuisine_description, inspection_dt, violation_code,
    violation_description, critical_flag, score_raw, score_int, grade, boro_raw,
    CASE
      WHEN zip_raw RLIKE '^[0-9]{5}$' AND zip_raw != '12345' THEN zip_raw
      WHEN zip_raw RLIKE '^[0-9]{5}-[0-9]{4}$' AND SUBSTR(zip_raw, 1, 5) != '12345' THEN SUBSTR(zip_raw, 1, 5)
      WHEN zip_raw RLIKE '^[0-9]{5}\\.0$' AND SUBSTR(zip_raw, 1, 5) != '12345' THEN SUBSTR(zip_raw, 1, 5)
      ELSE NULL
    END AS zip,
    CASE
      WHEN zip_raw IS NULL THEN 'blank'
      WHEN zip_raw = '12345' THEN 'suspicious_12345'
      WHEN zip_raw RLIKE '^[0-9]{5}(-[0-9]{4})?$' OR zip_raw RLIKE '^[0-9]{5}\\.0$' THEN 'valid'
      ELSE 'invalid_format'
    END AS dq_zip_flag,
    CASE
      WHEN boro_raw = '0' OR boro_raw IS NULL THEN 'unknown'
      WHEN UPPER(boro_raw) IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND') THEN 'valid'
      ELSE 'unexpected'
    END AS dq_boro_flag,
    CASE
      WHEN score_int IS NULL AND score_raw IS NOT NULL THEN 'parse_fail'
      WHEN score_int IS NULL THEN 'blank'
      ELSE 'valid'
    END AS dq_score_flag,
    CASE WHEN violation_code IS NULL THEN 'blank' ELSE 'present' END AS dq_violation_code_flag,
    CASE WHEN grade IS NULL THEN 'blank' ELSE 'present' END AS dq_grade_flag
  FROM parsed
),
-- Geography gate: drop NJ / Long Island / Westchester ZIPs (80 rows, 15 camis, all boro = 0 in this packet).
-- Logged in silver_dropped_non_nyc. Blank ZIP (zip IS NULL, 1513 rows) is kept — unknown is not out of state.
nyc_only AS (
  SELECT *
  FROM normalized
  WHERE zip IS NULL
     OR workspace.default.zip_region(zip, boro_raw) = 'nyc'
),
with_lookup AS (
  SELECT
    n.*,
    l.borough AS borough_from_zip,
    l.n_distinct_borough
  FROM nyc_only n
  LEFT JOIN workspace.default.silver_zip_boro_lookup l
    ON n.zip = l.zip AND l.rn = 1
),
filled AS (
  SELECT
    camis, dba, cuisine_description, inspection_dt, violation_code,
    violation_description, critical_flag, score_raw, score_int, grade,
    zip, dq_zip_flag, dq_boro_flag, dq_score_flag,
    dq_violation_code_flag, dq_grade_flag,
    CASE
      WHEN UPPER(boro_raw) IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND')
        THEN UPPER(boro_raw)
      WHEN n_distinct_borough = 1 THEN borough_from_zip
      ELSE NULL
    END AS borough,
    CASE
      WHEN UPPER(boro_raw) IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND')
        THEN 'source'
      WHEN n_distinct_borough = 1 THEN 'zip_lookup'
      WHEN n_distinct_borough > 1 THEN 'ambiguous_zip'
      ELSE 'unknown'
    END AS borough_source,
    CASE
      WHEN zip IS NULL THEN 'unknown'
      WHEN zip IN ('11430', '11371') THEN 'airport'
      WHEN zip = '10128' THEN 'neighborhood'
      WHEN zip LIKE '101%' THEN 'building'
      WHEN n_distinct_borough IS NULL THEN 'non_nyc'
      ELSE 'neighborhood'
    END AS zip_type
  FROM with_lookup
),
deduped AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY camis, dba, borough, zip, cuisine_description, inspection_dt,
                   violation_code, violation_description, critical_flag, score_int, grade
      ORDER BY inspection_dt
    ) AS rn
  FROM filled
),
score_conflicts AS (
  SELECT
    camis, inspection_dt,
    CASE
      WHEN COUNT(DISTINCT COALESCE(CAST(score_int AS STRING), '<blank>')) > 1
        THEN 'conflict' ELSE 'no_conflict'
    END AS dq_score_conflict
  FROM deduped
  WHERE rn = 1 AND camis IS NOT NULL AND inspection_dt IS NOT NULL
  GROUP BY camis, inspection_dt
)
SELECT
  d.camis, d.dba, d.borough, d.borough_source, d.zip, d.zip_type,
  d.cuisine_description, d.inspection_dt,
  d.violation_code, d.violation_description, d.critical_flag,
  d.score_raw, d.score_int, d.grade,
  d.dq_zip_flag, d.dq_boro_flag, d.dq_score_flag,
  d.dq_violation_code_flag, d.dq_grade_flag,
  COALESCE(sc.dq_score_conflict, 'no_conflict') AS dq_score_conflict,
  'restaurant_inspections.csv' AS _source_file,
  current_timestamp() AS _ingested_at
FROM deduped d
LEFT JOIN score_conflicts sc
  ON d.camis = sc.camis AND d.inspection_dt = sc.inspection_dt
WHERE d.rn = 1;

COMMENT ON TABLE workspace.default.silver_restaurant_inspections IS
  'Cleaned DOHMH rows for NYC ZIPs only. GRAIN IS STILL ONE VIOLATION — do not COUNT(*) or SUM(score) as inspections. 166 exact duplicate extras removed. Blank ZIP kept. boro 0 nulled. NJ / Long Island / Westchester rows (80 rows, 15 camis) REMOVED via zip_region() and logged in silver_dropped_non_nyc. 04K rat, 04L mouse, 08A harborage. Grade A row count coincidentally equals 311 ticket count (50954) — coincidence. Use silver_inspection_visits for visit grain. Kitchen inspections are not a 311 response; no ticket id, no inspection_type.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.camis IS
  'Establishment ID. Count restaurants with COUNT(DISTINCT camis), never by rows or dba. This is not a complete restaurant registry.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.dba IS
  'Trade name; not unique. Do not count restaurants by dba.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.borough IS
  'NYC borough. boro 0 nulled; filled from ZIP only when that ZIP has exactly one NYC borough in the packet.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.borough_source IS
  'source / zip_lookup / ambiguous_zip / unknown.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.zip IS
  'Normalized 5-digit ZIP. Blank ZIP kept as NULL (1513 rows). Do not impute. Not a block.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.zip_type IS
  'neighborhood / airport (11430 JFK, 11371 LGA) / building (101xx except 10128) / unknown. 10128 is UES neighborhood, not a building. non_nyc is a guard only (expect 0): NJ / Long Island / Westchester rows are removed upstream by zip_region(). Airports: kitchen coverage only, no SGI rank.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.cuisine_description IS
  'Optional footnote. Do not put cuisine in SGI unless you can defend it. Bodegas vs offices collect pest codes differently.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.inspection_dt IS
  'Inspection date at midnight. (camis, inspection_dt) is an establishment-date PROXY, not a guaranteed unique inspection. No inspection_type in the packet.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.violation_code IS
  'Exact code. 04K = rat evidence, 04L = mouse evidence, 08A = harborage not a live rat. Do not equate Critical with rodents. Blank is not a clean inspection.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.violation_description IS
  'Inspector text for the code. Do not parse species from free text.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.critical_flag IS
  'Critical vs not. Critical is not the same as rodent evidence.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.score_raw IS
  'Raw score string before cast. May repeat across rows of the same visit.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.score_int IS
  'Parsed score. Never SUM(score) at violation grain (inflates ~5x). Carry score once per visit in silver_inspection_visits. Grade A rows coincidentally equal 50954 — do not pitch that match.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.grade IS
  'Often blank. Historical A/B/C/N/P/Z is not current official grade. May repeat across rows of the same visit.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.dq_zip_flag IS
  'valid / blank / suspicious_12345 / invalid_format.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.dq_boro_flag IS
  'valid / unknown (including boro 0) / unexpected.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.dq_score_flag IS
  'valid / blank / parse_fail. Parse failures are not zeros.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.dq_violation_code_flag IS
  'present / blank. Keep blank-code rows in the observed establishment universe.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.dq_grade_flag IS
  'present / blank.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections.dq_score_conflict IS
  'conflict if the same camis+date has disagreeing score_int, including blank vs populated (~4621 visits). Flagged, not averaged.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections._source_file IS
  'Packet filename: restaurant_inspections.csv.';
COMMENT ON COLUMN workspace.default.silver_restaurant_inspections._ingested_at IS
  'Table rebuild timestamp. Not an inspection as-of cutoff.';

In [0]:
%sql
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT camis) AS restaurants,
  (SELECT COUNT(*) FROM (
    SELECT DISTINCT camis, inspection_dt
    FROM workspace.default.silver_restaurant_inspections
    WHERE camis IS NOT NULL AND inspection_dt IS NOT NULL
  ) t) AS visits,
  SUM(CASE WHEN zip IS NULL THEN 1 ELSE 0 END) AS n_null_zip,
  SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) AS n_null_borough,
  SUM(CASE WHEN borough_source = 'zip_lookup' THEN 1 ELSE 0 END) AS n_boro_from_zip,
  SUM(CASE WHEN zip_type = 'non_nyc' THEN 1 ELSE 0 END) AS n_non_nyc,
  SUM(CASE WHEN zip_type = 'airport' THEN 1 ELSE 0 END) AS n_airport,
  SUM(CASE WHEN zip_type = 'building' THEN 1 ELSE 0 END) AS n_building,
  SUM(CASE WHEN dq_score_flag = 'parse_fail' THEN 1 ELSE 0 END) AS n_score_parse_fail,
  SUM(CASE WHEN violation_code = '04K' THEN 1 ELSE 0 END) AS n_04k,
  SUM(CASE WHEN violation_code = '04L' THEN 1 ELSE 0 END) AS n_04l,
  SUM(CASE WHEN violation_code = '08A' THEN 1 ELSE 0 END) AS n_08a,
  SUM(CASE WHEN grade = 'A' THEN 1 ELSE 0 END) AS n_grade_a,
  (SELECT COUNT(*) FROM (
    SELECT DISTINCT camis, inspection_dt
    FROM workspace.default.silver_restaurant_inspections
    WHERE dq_score_conflict = 'conflict'
  ) t) AS n_score_conflict_pairs
FROM workspace.default.silver_restaurant_inspections;

n_rows,restaurants,visits,n_null_zip,n_null_borough,n_boro_from_zip,n_non_nyc,n_airport,n_building,n_score_parse_fail,n_04k,n_04l,n_08a,n_grade_a,n_score_conflict_pairs
157837,26099,44423,1513,15,17,0,320,725,1,1502,6948,12485,50923,4621


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_inspection_visits AS
SELECT
  camis,
  inspection_dt,
  MAX(dba) AS dba,
  CASE WHEN COUNT(DISTINCT zip) = 1 THEN MAX(zip) END AS zip,
  CASE WHEN COUNT(DISTINCT zip) = 1 THEN MAX(zip_type) END AS zip_type,
  CASE WHEN COUNT(DISTINCT borough) = 1 THEN MAX(borough) END AS borough,
  CASE WHEN COUNT(DISTINCT borough_source) = 1 THEN MAX(borough_source) ELSE 'mixed' END AS borough_source,
  MAX(cuisine_description) AS cuisine_description,
  MIN(score_int) AS score_min,
  MAX(score_int) AS score_max,
  CASE WHEN COUNT(DISTINCT grade) = 1 THEN MAX(grade) END AS grade,
  COUNT(*) AS n_violation_rows,
  SUM(CASE WHEN violation_code = '04K' THEN 1 ELSE 0 END) AS n_04k_rats,
  SUM(CASE WHEN violation_code = '04L' THEN 1 ELSE 0 END) AS n_04l_mice,
  MAX(CASE WHEN violation_code IN ('04K', '04L') THEN 1 ELSE 0 END) AS has_rodent_violation,
  SUM(CASE WHEN violation_code = '08A' THEN 1 ELSE 0 END) AS n_08a_harborage,
  MAX(CASE WHEN violation_code = '08A' THEN 1 ELSE 0 END) AS has_08a_harborage,
  SUM(CASE WHEN dq_violation_code_flag = 'blank' THEN 1 ELSE 0 END) AS n_blank_violation_code,
  MAX(CASE WHEN dq_violation_code_flag = 'blank' THEN 1 ELSE 0 END) AS has_blank_violation_code,
  MAX(CASE WHEN dq_score_conflict = 'conflict' THEN 1 ELSE 0 END) AS score_conflict_flag,
  MAX(CASE WHEN zip IS NULL THEN 1 ELSE 0 END) AS zip_missing_on_any_row,
  current_timestamp() AS _ingested_at
FROM workspace.default.silver_restaurant_inspections
WHERE camis IS NOT NULL AND inspection_dt IS NOT NULL
GROUP BY camis, inspection_dt;

COMMENT ON TABLE workspace.default.silver_inspection_visits IS
  'One row per restaurant visit (camis + inspection_date proxy). Score is a min/max window; do not treat score_max as truth when score_conflict_flag=1. has_rodent_violation is 04K (rats) or 04L (mice) only. n_08a_harborage / has_08a_harborage are separate — 08A is not a live rat. Blank violation codes are unknown, not clean. This is kitchen evidence of places DOHMH already visits, not 311 demand and not a response to a complaint. No shared ticket id.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.camis IS
  'Establishment ID. Count restaurants with COUNT(DISTINCT camis). Observed inspected places only — not all food on the block.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.inspection_dt IS
  'Establishment-date proxy, not a guaranteed unique inspection event. No inspection_type. An inspection on Tuesday is not the response to Monday 311.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.dba IS
  'Trade name from MAX(dba) at this visit. Not unique.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.zip IS
  'ZIP when every finding row at this visit agrees; otherwise NULL. Blank ZIP visits stay NULL.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.zip_type IS
  'neighborhood / airport / building / unknown when ZIP is consistent across rows. non_nyc is a guard only — out-of-city visits are removed upstream by zip_region().';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.borough IS
  'Borough when every finding row agrees; otherwise NULL.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.borough_source IS
  'source / zip_lookup / mixed / unknown. mixed means rows at this visit disagree.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.cuisine_description IS
  'MAX cuisine at this visit. Optional footnote; not an SGI input.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.score_min IS
  'MIN(score_int) across finding rows. Blanks ignored, so min can equal max even when score_conflict_flag = 1.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.score_max IS
  'MAX(score_int) across finding rows. Do not treat as truth when score_conflict_flag = 1. Never SUM scores.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.grade IS
  'Set only when every finding row at this visit has the same grade. NULL if they disagree. Not current official grade.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.n_violation_rows IS
  'Finding rows at this visit. Dirty kitchens manufacture more rows. Not an inspection count and not a rodent count.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.n_04k_rats IS
  '04K rat-evidence finding rows at this visit. Direct rat evidence, not 08A.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.n_04l_mice IS
  '04L mouse-evidence finding rows at this visit. Indoor mice are not street rats.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.has_rodent_violation IS
  '1 if any 04K or 04L at this visit. Direct rodent evidence. Kitchen coverage context, not 311 demand, not a complaint response.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.n_08a_harborage IS
  '08A finding rows. Conditions conducive to rodents, insects, or other pests — not a live rat. Keep separate from has_rodent_violation.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.has_08a_harborage IS
  '1 if any 08A at this visit. Harborage, not a live rat. Do not fold into has_rodent_violation.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.n_blank_violation_code IS
  'Finding rows with no violation code. Missing code is not proof of a clean inspection. Keep in the establishment universe.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.has_blank_violation_code IS
  '1 if any blank violation code at this visit. Unknown finding status, not a pass.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.score_conflict_flag IS
  '1 if this camis+date had disagreeing raw scores (including blank vs populated). About 4621 visits.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits.zip_missing_on_any_row IS
  '1 if any finding row at this visit has NULL ZIP.';
COMMENT ON COLUMN workspace.default.silver_inspection_visits._ingested_at IS
  'Table rebuild timestamp.';

In [0]:
%sql
SELECT
  COUNT(*) AS visits,
  COUNT(DISTINCT camis) AS restaurants,
  SUM(has_rodent_violation) AS visits_with_rodent_vio,
  SUM(has_08a_harborage) AS visits_with_08a,
  SUM(has_blank_violation_code) AS visits_with_blank_code,
  SUM(score_conflict_flag) AS visits_score_conflict,
  SUM(zip_missing_on_any_row) AS visits_missing_zip,
  SUM(CASE WHEN zip_type = 'airport' THEN 1 ELSE 0 END) AS visits_airport,
  SUM(CASE WHEN zip_type = 'non_nyc' THEN 1 ELSE 0 END) AS visits_non_nyc
FROM workspace.default.silver_inspection_visits;

visits,restaurants,visits_with_rodent_vio,visits_with_08a,visits_with_blank_code,visits_score_conflict,visits_missing_zip,visits_airport,visits_non_nyc
44423,26099,8044,12485,1795,4621,461,142,0


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_zip_spine AS
WITH r AS (
  SELECT
    zip,
    COUNT(*) AS n_311,
    MAX(CASE WHEN cluster_key IS NOT NULL THEN cluster_n END) AS max_cluster_n,
    COUNT(DISTINCT cluster_key) AS n_distinct_clusters,
    MAX(zip_type) AS zip_type_311
  FROM workspace.default.silver_rat_sightings
  WHERE zip IS NOT NULL
  GROUP BY zip
),
v AS (
  SELECT
    zip,
    COUNT(*) AS n_visits,
    COUNT(DISTINCT camis) AS n_restaurants,
    SUM(has_rodent_violation) AS n_visits_rodent_vio,
    MAX(zip_type) AS zip_type_kit
  FROM workspace.default.silver_inspection_visits
  WHERE zip IS NOT NULL
  GROUP BY zip
),
keys AS (
  SELECT zip FROM r
  UNION
  SELECT zip FROM v
)
SELECT
  k.zip,
  CASE WHEN l.n_distinct_borough = 1 THEN l.borough END AS borough,
  l.borough AS borough_mode,
  CASE
    WHEN l.n_distinct_borough > 1 THEN TRUE
    ELSE FALSE
  END AS borough_ambiguous,
  CASE
    WHEN k.zip IN ('11430', '11371') THEN 'airport'
    WHEN k.zip = '10128' THEN 'neighborhood'
    WHEN k.zip LIKE '101%' THEN 'building'
    WHEN l.zip IS NULL THEN 'non_nyc'
    WHEN COALESCE(r.n_311, 0) < 10 THEN 'thin'
    ELSE 'neighborhood'
  END AS zip_type,
  COALESCE(r.n_311, 0) AS n_311,
  COALESCE(v.n_visits, 0) AS n_visits,
  COALESCE(v.n_restaurants, 0) AS n_restaurants,
  COALESCE(v.n_visits_rodent_vio, 0) AS n_visits_rodent_vio,
  r.max_cluster_n,
  r.n_distinct_clusters,
  CASE
    WHEN COALESCE(r.n_311, 0) = 0 THEN NULL
    ELSE ROUND(r.max_cluster_n / r.n_311, 3)
  END AS max_cluster_share,
  CASE
    WHEN COALESCE(r.n_311, 0) >= 30
         AND r.max_cluster_n * 1.0 / r.n_311 > 0.30 THEN TRUE
    ELSE FALSE
  END AS is_cluster_dominant,
  current_timestamp() AS _ingested_at
FROM keys k
LEFT JOIN workspace.default.silver_zip_boro_lookup l
  ON k.zip = l.zip AND l.rn = 1
LEFT JOIN r ON k.zip = r.zip
LEFT JOIN v ON k.zip = v.zip;

COMMENT ON TABLE workspace.default.silver_zip_spine IS
  'One row per NYC ZIP seen in 311 or kitchens (observed coverage, not every NYC ZIP). NJ / Long Island / Westchester ZIPs are removed upstream by zip_region() and logged in silver_dropped_non_nyc. zip_type: neighborhood / airport (11430 JFK, 11371 LGA) / building (101xx except 10128) / thin (n_311<10); non_nyc is a guard only (expect 0). 10128 is neighborhood (UES). Do not rank airport/non_nyc/building/thin. is_cluster_dominant requires n_311 >= 30 and max_cluster_share > 0.30 — 10035 is one GPS cluster (~0.817), a known outreach ZIP, not a cursed block. n_restaurants is inspected establishments, not population. Gold computes SGI-30 from tickets; this spine is geography and denominators only. Closed percent is not SGI.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.zip IS
  'Observed ZIP from 311 or kitchens. Not every NYC ZIP. A missing lookup ZIP is not represented in this extract, not automatically invalid. This is a ZIP, not a block.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.borough IS
  'Filled only when the packet has exactly one NYC borough for this ZIP. NULL if ambiguous (10463, 11370).';
COMMENT ON COLUMN workspace.default.silver_zip_spine.borough_mode IS
  'Modal borough even when ambiguous. Do not treat as a resolved borough when borough_ambiguous is true.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.borough_ambiguous IS
  'TRUE for 10463 (Marble Hill) and 11370 (Rikers / East Elmhurst). Do not pick a side silently.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.zip_type IS
  'neighborhood / airport (11430 JFK, 11371 LGA) / building (101xx except 10128) / thin (n_311 < 10). 10128 is a neighborhood (UES), not a building. non_nyc is a guard only — NJ / Long Island / Westchester ZIPs are removed upstream by zip_region() (see silver_dropped_non_nyc). Do not rank airport, building, or thin as a cursed block. Airports: kitchen coverage only.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.n_311 IS
  'Distinct 311 tickets with this ZIP (0 if kitchen-only). Zero requests is not zero rats and not fine service. Not SGI. Quiet ZIPs may have given up on calling.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.n_visits IS
  'Establishment-date proxies in this ZIP. Of restaurants in 2025-now inspections, not all restaurants.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.n_restaurants IS
  'COUNT(DISTINCT camis) in this ZIP. Denominator because population is not in the packet. Punishes residential ZIPs vs kitchen-dense ones — pair with zip_type. 0 means restaurant rates are undefined, not zero.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.n_visits_rodent_vio IS
  'Visits with 04K or 04L. Parallel DOHMH kitchen program, not a 311 response. Do not divide violations by complaints.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.max_cluster_n IS
  'Tickets at the busiest GPS cluster in this ZIP. 10035 is ~1227 at one cluster_key.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.n_distinct_clusters IS
  'Unique GPS cluster_key values. Density proxy from the packet (no population file).';
COMMENT ON COLUMN workspace.default.silver_zip_spine.max_cluster_share IS
  'max_cluster_n / n_311. 10035 ~0.817. One site, not 1501 independent rats. Extra 311 in 10035 can mean outreach (rat mitigation zone), not a unique infestation.';
COMMENT ON COLUMN workspace.default.silver_zip_spine.is_cluster_dominant IS
  'TRUE only if n_311 >= 30 AND one GPS cluster is more than 30pct of tickets. Then say one site, not cursed ZIP. Thin n is not cluster-dominant. 10035 is the stress test of the metric, not the villain.';
COMMENT ON COLUMN workspace.default.silver_zip_spine._ingested_at IS
  'Table rebuild timestamp. SGI-30 is not stored here; gold computes it from silver_rat_sightings.';

In [0]:
%sql
SELECT zip, borough, borough_mode, borough_ambiguous, zip_type, n_311, n_visits, n_restaurants, max_cluster_share, n_distinct_clusters, is_cluster_dominant
FROM workspace.default.silver_zip_spine
WHERE zip IN ('10035', '11430', '11371', '12345', '07307', '11701', '10116', '10128', '10463', '11370', '11237', '11103', '10020')
ORDER BY zip;

zip,borough,borough_mode,borough_ambiguous,zip_type,n_311,n_visits,n_restaurants,max_cluster_share,n_distinct_clusters,is_cluster_dominant
10020,MANHATTAN,MANHATTAN,false,thin,1,58,40,1.0,1,false
10035,MANHATTAN,MANHATTAN,false,neighborhood,1501,153,83,0.817,139,true
10116,null,null,false,building,0,3,3,null,null,false
10128,MANHATTAN,MANHATTAN,false,neighborhood,531,233,146,0.081,188,false
10463,null,BRONX,true,neighborhood,465,214,122,0.082,210,false
11103,QUEENS,QUEENS,false,neighborhood,828,447,240,0.545,153,true
11237,BROOKLYN,BROOKLYN,false,neighborhood,502,411,257,0.028,274,false
11370,null,QUEENS,true,neighborhood,106,28,17,0.075,68,false
11371,QUEENS,QUEENS,false,airport,0,8,8,null,null,false
11430,QUEENS,QUEENS,false,airport,2,134,88,0.5,2,false


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.default.silver_rat_sightings
    WHERE zip IS NOT NULL AND workspace.default.zip_region(zip, borough) <> 'nyc') AS tickets_non_nyc,
  (SELECT COUNT(*) FROM workspace.default.silver_restaurant_inspections
    WHERE zip IS NOT NULL AND workspace.default.zip_region(zip, borough) <> 'nyc') AS kitchen_rows_non_nyc,
  (SELECT COUNT(*) FROM workspace.default.silver_zip_spine
    WHERE zip_type = 'non_nyc') AS spine_non_nyc,
  (SELECT COUNT(*) FROM workspace.default.silver_dropped_non_nyc) AS dropped_rows_logged;

tickets_non_nyc,kitchen_rows_non_nyc,spine_non_nyc,dropped_rows_logged
0,0,0,80


In [0]:
%sql
SELECT zip_type, COUNT(*) AS n_zips, SUM(n_311) AS tickets, SUM(n_visits) AS visits,
  SUM(CASE WHEN is_cluster_dominant THEN 1 ELSE 0 END) AS n_cluster_dominant
FROM workspace.default.silver_zip_spine
GROUP BY zip_type
ORDER BY n_zips DESC;

zip_type,n_zips,tickets,visits,n_cluster_dominant
neighborhood,172,50904,43306,6
building,30,8,271,0
thin,18,39,243,0
airport,2,2,142,0


## Silver contract (paste into JUDGMENT_CALLS)

- **311:** 50,954 tickets kept. 1 ZIP (`12345`) quarantined to NULL. 94 missing GPS flagged, not imputed. 1 Unspecified borough + ZIP `11237` filled from ZIP (`borough_source='zip_lookup'`).
  - **descriptor_category** splits the four descriptors: `rat_sighting` (32,942), `condition_attracting_rodents` (10,788), `signs_of_rodents` (5,178), `mouse_sighting` (2,046). Indoor mice ≠ street rats — Gold must not match 04K kitchens to mouse-in-apt 311.
  - **location_category** groups 22 location types: `dwelling` (~77.5%), `public_space`, `commercial`, `infrastructure`, `institution`, `vacant`, `other`. 3+ Family Apt is 49.8% of all tickets.
  - **is_same_timestamp_close:** ~22,547 tickets have `closed_ts = created_ts`. Closed is a clock, not a visit. Resolution text is not in the packet. **Do not put Closed% in SGI.**
- **Geography:** NYC ZIPs only. `workspace.default.zip_region()` removes New Jersey (`07`/`08`), Long Island Nassau/Suffolk (`115`, `117`–`119`) and Westchester/Rockland (`105`–`109`) rows from every silver table; `110xx` counts as NYC only with an NYC borough label. Dropped rows are logged in `silver_dropped_non_nyc`: **80 kitchen rows, 15 establishments, 24 visits, 12 ZIPs, 0 tickets** — all `boro=0` (NYC-permitted vendors with an out-of-state commissary/HQ address, e.g. Brooklyn Crab in 07304). Blank ZIP is **not** dropped.
- **Kitchens:** 157,837 violation rows (166 exact-dup extras removed, 80 out-of-city rows removed). **26,099** restaurants, **44,423** visits. Blank ZIP kept. `boro=0` nulled.
  - **04K / 04L** = direct rodent evidence (`has_rodent_violation`). **08A** = harborage, not a live rat (`has_08a_harborage`). Blank violation code is unknown, not clean.
  - Grade A violation rows coincidentally equal 50,954 (the 311 count). Coincidence. Never pitch it. Never `SUM(score)` at violation grain.
- **Do not fill:** ZIP, lat/long, grade, score (conflicts flagged). **Do fill:** borough from ZIP when unique in-packet.
- **Do not:** join 311 to kitchens into one fact table; call `Closed` inspected; rank JFK/LGA/`101xx` (except 10128) as a cursed neighborhood; fetch ACS; re-admit NJ/LI rows by loosening `zip_region()`.
- **Spine:** observed NYC ZIP coverage from both sources (234 → 222 ZIPs after the geography gate), not every NYC ZIP. `is_cluster_dominant` requires **n_311 ≥ 30** and one GPS cluster **>30%** of tickets. 10035 `max_cluster_share` ≈ 0.817 — say "one site," not "cursed ZIP." Treat 10035 as a **stress test** (known rat-mitigation / outreach ZIP). `zip_type` separates `neighborhood` / `airport` / `building` / `non_nyc` / `thin` (n_311<10).

### SGI contract (gold computes this; silver does not store a score)

The question is: when someone calls for help, does the city show up? **These files cannot prove a visit.** There is no resolution_description, no shared ticket/inspection id, and no inspection_type.

**Official index: SGI-30**, "30-day administrative closure gap," scale 0–100, **higher = larger share without a recorded closure within 30 days**. This is a **service-process proxy**, not proof the city failed to visit, and not a legal SLA.

For each ZIP and the full creation window, using `silver_rat_sightings` (not spine `n_311`):

1. **A** = inferred observation cutoff = `MAX(created_ts)` on validated 311 (not `current_timestamp()`).
2. Eligible tickets: unique_key, complaint_type = Rodent (all four descriptors), valid `created_ts`, ZIP on the eligible geography, `created_ts <= A - INTERVAL 30 DAYS`.
3. Exclude `dq_closed_flag` in (`parse_fail`, `negative_duration`) from N and K; count them separately.
4. **N** = eligible mature tickets. **K** = those with `closed_ts` at or after created, at or before created + 30 days, and at or before A.
5. **SGI-30** = `100.0 * (N - K) / NULLIF(N, 0)`. N=0 → NULL, never 0 or 100.
6. Tickets younger than 30 days are in neither N nor K; show them separately. Never count yesterday's In Progress as delayed service.
7. **Display:** if `1 <= N < 30`, show counts and "insufficient data"; **suppress the published score** and exclude from ranked comparisons. If `N >= 30`, show the score without claiming sample size removes reporting bias.
8. **Rank set:** `zip_type = neighborhood` on the spine AND N ≥ 30. Airports, `101xx` except **10128**, thin: lookup card only. NJ/LI ZIPs are not on the spine at all — a user typing 07307 gets "not an NYC ZIP", not a card. Compute citywide ranks **before** filtering to a user ZIP.
9. City/borough rate = `100 * SUM(N-K) / SUM(N)` over all eligible mature tickets, including ZIPs suppressed on the leaderboard.

**Put beside SGI-30, not inside it:** observed establishments, 04K share, 04L share, 08A share (separate), `n_311`, `n_distinct_clusters`, `max_cluster_share` / `is_cluster_dominant`, descriptor split, location split, same-timestamp close count, In Progress older than ~14 days (backlog). Kitchen evidence is a parallel program, not a causal 311 response.

**Do not:** use Closed% of all tickets as SGI; crown 10035 "best served" because SGI-30 is tiny (instant closes + one GPS cluster + outreach); crown a quiet ZIP "fine"; divide violation rows by complaints; join raw 311 to raw kitchens on ZIP.